# Build Texas ZCTA to PUMA Crosswalk

This notebook builds a Texas ZIP/ZCTA -> 2020 PUMA crosswalk from the official Census 2020 PUMA-to-ZCTA relationship file. The output is designed for later ACS PUMS demographic sampling for synthetic Texas postal customers.

## 1. Imports and Configuration

All paths are relative to this notebook's working directory. The notebook is restartable: it creates directories as needed, reuses downloaded source data, and overwrites processed outputs deterministically.

In [ ]:
from pathlib import Path
import csv
import os
import urllib.request

import numpy as np
import pandas as pd

CENSUS_URL = "https://www2.census.gov/geo/docs/maps-data/data/rel2020/puma520/tab20_puma520_zcta520_natl.txt"
SOURCE_FILE = "tab20_puma520_zcta520_natl.txt"
SOURCE_YEAR = 2020

RAW_DIR = Path("data/raw/census")
PROCESSED_DIR = Path("data/processed/census")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILE_PATH = RAW_DIR / SOURCE_FILE
MAIN_OUTPUT_PATH = PROCESSED_DIR / "tx_zcta_to_puma_crosswalk_2020.csv"
AUDIT_OUTPUT_PATH = PROCESSED_DIR / "tx_zcta_to_puma_crosswalk_2020_audit.csv"

TARGET_WEIGHT_CANDIDATES = [
    Path("texas_target_weights.csv"),
    Path("data/raw/census/texas_target_weights.csv"),
    Path("../../Establish_source_codes_and_target_weights/texas_target_weights.csv"),
    Path("Establish_source_codes_and_target_weights/texas_target_weights.csv"),
]

RAW_COLUMNS = [
    "GEOID_PUMA5_20",
    "NAMELSAD_PUMA5_20",
    "AREALAND_PUMA5_20",
    "AREAWATER_PUMA5_20",
    "GEOID_ZCTA5_20",
    "NAMELSAD_ZCTA5_20",
    "AREALAND_ZCTA5_20",
    "AREAWATER_ZCTA5_20",
    "AREALAND_PART",
    "AREAWATER_PART",
]

AREA_COLUMNS = [
    "AREALAND_PUMA5_20",
    "AREAWATER_PUMA5_20",
    "AREALAND_ZCTA5_20",
    "AREAWATER_ZCTA5_20",
    "AREALAND_PART",
    "AREAWATER_PART",
]

FINAL_COLUMNS = [
    "zcta",
    "state_fips",
    "puma_geoid",
    "puma_code",
    "puma_name",
    "zcta_name",
    "zcta_land_area",
    "puma_land_area",
    "overlap_land_area",
    "overlap_water_area",
    "zcta_to_puma_weight",
    "puma_to_zcta_weight",
    "overlap_rank_for_zcta",
    "is_primary_puma_for_zcta",
    "source_year",
    "source_file",
]

ZIP_COLUMN_CANDIDATES = ["zip_code", "zip", "zcta", "assigned_zip", "ZIP", "ZCTA"]

## 2. Download Census Relationship File

The official Census relationship file is downloaded only when it is not already present locally.

In [ ]:
def download_if_needed(url: str, destination: Path) -> Path:
    if destination.exists():
        print(f"Reusing existing file: {destination}")
        return destination

    print(f"Downloading Census relationship file to: {destination}")
    urllib.request.urlretrieve(url, destination)
    return destination


def detect_delimiter(path: Path) -> str:
    with path.open("r", encoding="utf-8", errors="replace") as file:
        sample = file.read(8192)
    try:
        dialect = csv.Sniffer().sniff(sample, delimiters="|,\t")
        return dialect.delimiter
    except csv.Error:
        header = sample.splitlines()[0]
        counts = {delimiter: header.count(delimiter) for delimiter in ["|", ",", "\t"]}
        return max(counts, key=counts.get)


def load_relationship_file(path: Path) -> pd.DataFrame:
    delimiter = detect_delimiter(path)
    print(f"Detected delimiter: {repr(delimiter)}")

    df = pd.read_csv(path, sep=delimiter, dtype="string", low_memory=False)
    df.columns = df.columns.str.strip()

    missing_columns = sorted(set(RAW_COLUMNS) - set(df.columns))
    if missing_columns:
        raise ValueError(f"Missing expected Census columns: {missing_columns}")

    df = df[RAW_COLUMNS].copy()
    for column in AREA_COLUMNS:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    file_size_mb = path.stat().st_size / (1024 * 1024)
    print(f"File path: {path}")
    print(f"File size: {file_size_mb:,.2f} MB")
    print(f"Loaded row count: {len(df):,}")
    return df


download_if_needed(CENSUS_URL, RAW_FILE_PATH)
raw_relationship = load_relationship_file(RAW_FILE_PATH)

## 3. Normalize IDs and Filter to Texas

Census GEOIDs must stay as strings so leading zeros are preserved. PUMA GEOIDs are 7 characters: 2-character state FIPS plus 5-character PUMA code.

In [ ]:
def normalize_ids(df: pd.DataFrame) -> pd.DataFrame:
    normalized = df.copy()

    # Keep Census identifiers as strings so leading zeros survive joins and API calls.
    normalized["puma_geoid"] = normalized["GEOID_PUMA5_20"].astype("string").str.strip().str.zfill(7)
    normalized["state_fips"] = normalized["puma_geoid"].str[:2]
    normalized["puma_code"] = normalized["puma_geoid"].str[-5:]
    normalized["zcta"] = normalized["GEOID_ZCTA5_20"].astype("string").str.strip().str.zfill(5)
    normalized["puma_name"] = normalized["NAMELSAD_PUMA5_20"].astype("string").str.strip()
    normalized["zcta_name"] = normalized["NAMELSAD_ZCTA5_20"].astype("string").str.strip()

    assert normalized["puma_geoid"].str.len().eq(7).all(), "puma_geoid must be 7 characters"
    assert normalized["state_fips"].str.len().eq(2).all(), "state_fips must be 2 characters"
    assert normalized["puma_code"].str.len().eq(5).all(), "puma_code must be 5 characters"
    assert normalized["zcta"].str.len().eq(5).all(), "zcta must be 5 characters"
    return normalized


def find_existing_target_weights(candidates: list[Path]) -> Path | None:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def infer_zip_column(columns: pd.Index) -> str | None:
    exact_lookup = {column.lower(): column for column in columns}
    for candidate in ZIP_COLUMN_CANDIDATES:
        if candidate.lower() in exact_lookup:
            return exact_lookup[candidate.lower()]
    return None


def load_target_zctas() -> tuple[set[str] | None, Path | None, str | None]:
    target_path = find_existing_target_weights(TARGET_WEIGHT_CANDIDATES)
    if target_path is None:
        print("No texas_target_weights.csv file found. Proceeding with all Texas ZCTAs.")
        return None, None, None

    target_weights = pd.read_csv(target_path, dtype="string", low_memory=False)
    zip_column = infer_zip_column(target_weights.columns)
    if zip_column is None:
        print(f"Found {target_path}, but no ZIP/ZCTA column was recognized. Proceeding with all Texas ZCTAs.")
        return None, target_path, None

    target_zctas = set(target_weights[zip_column].dropna().astype("string").str.strip().str.zfill(5))
    print(f"Using target ZIP filter from: {target_path}")
    print(f"Inferred ZIP/ZCTA column: {zip_column}")
    print(f"Target ZIP/ZCTA count: {len(target_zctas):,}")
    return target_zctas, target_path, zip_column


normalized_relationship = normalize_ids(raw_relationship)
texas_relationship_all = normalized_relationship.loc[normalized_relationship["state_fips"].eq("48")].copy()

target_zctas, target_weights_path, target_zip_column = load_target_zctas()
if target_zctas is None:
    texas_relationship = texas_relationship_all.copy()
    used_target_filter = False
else:
    texas_relationship = texas_relationship_all.loc[texas_relationship_all["zcta"].isin(target_zctas)].copy()
    used_target_filter = True

print(f"Raw rows: {len(raw_relationship):,}")
print(f"Texas rows before optional target filtering: {len(texas_relationship_all):,}")
print(f"Rows after optional target filtering: {len(texas_relationship):,}")
print(f"Unique Texas ZCTAs after optional target filtering: {texas_relationship['zcta'].nunique():,}")
print(f"Unique Texas PUMAs after optional target filtering: {texas_relationship['puma_geoid'].nunique():,}")

## 4. Compute Weights and Rank Overlaps

`zcta_to_puma_weight` is the main sampling weight for later customer profile generation. It uses land overlap only, not water overlap.

In [ ]:
def prepare_positive_overlap_rows(df: pd.DataFrame) -> pd.DataFrame:
    prepared = df.copy()
    prepared["zcta_land_area"] = prepared["AREALAND_ZCTA5_20"]
    prepared["puma_land_area"] = prepared["AREALAND_PUMA5_20"]
    prepared["overlap_land_area"] = prepared["AREALAND_PART"]
    prepared["overlap_water_area"] = prepared["AREAWATER_PART"]
    return prepared.loc[prepared["overlap_land_area"].notna() & prepared["overlap_land_area"].gt(0)].copy()


texas_positive_all = prepare_positive_overlap_rows(texas_relationship_all)
crosswalk = prepare_positive_overlap_rows(texas_relationship)

zcta_overlap_total = crosswalk.groupby("zcta")["overlap_land_area"].transform("sum")
puma_overlap_total_all_tx = texas_positive_all.groupby("puma_geoid")["overlap_land_area"].sum()

crosswalk["zcta_to_puma_weight"] = crosswalk["overlap_land_area"] / zcta_overlap_total
crosswalk["puma_to_zcta_weight"] = crosswalk["overlap_land_area"] / crosswalk["puma_geoid"].map(puma_overlap_total_all_tx)

crosswalk = crosswalk.sort_values(["zcta", "overlap_land_area", "puma_geoid"], ascending=[True, False, True]).copy()
crosswalk["overlap_rank_for_zcta"] = crosswalk.groupby("zcta").cumcount() + 1
crosswalk["is_primary_puma_for_zcta"] = np.where(crosswalk["overlap_rank_for_zcta"].eq(1), 1, 0)

crosswalk["zcta_to_puma_weight"] = crosswalk["zcta_to_puma_weight"].round(6)
crosswalk["puma_to_zcta_weight"] = crosswalk["puma_to_zcta_weight"].round(6)
crosswalk["source_year"] = SOURCE_YEAR
crosswalk["source_file"] = SOURCE_FILE

final_crosswalk = crosswalk[FINAL_COLUMNS].copy()
audit_columns = FINAL_COLUMNS + [column for column in RAW_COLUMNS if column not in FINAL_COLUMNS]
audit_crosswalk = crosswalk[audit_columns].copy()

print(f"Positive-overlap output rows: {len(final_crosswalk):,}")
print(f"Output ZCTAs: {final_crosswalk['zcta'].nunique():,}")
print(f"Output PUMAs: {final_crosswalk['puma_geoid'].nunique():,}")

## 5. Data Validation

These checks fail loudly for structural problems. The reverse PUMA-to-ZCTA weight check warns instead of failing when a target-ZCTA filter is applied, because removed ZCTAs can legitimately make retained PUMA totals less than 1.

In [ ]:
def assert_no_missing_required_fields(df: pd.DataFrame, columns: list[str]) -> None:
    missing_counts = df[columns].isna().sum()
    missing_counts = missing_counts.loc[missing_counts.gt(0)]
    if not missing_counts.empty:
        raise AssertionError(f"Missing required final fields:\n{missing_counts}")


def validate_crosswalk(df: pd.DataFrame, used_filter: bool, tolerance: float = 1e-3) -> None:
    assert list(df.columns) == FINAL_COLUMNS, "Final columns are not in the required order"
    assert df["zcta"].astype("string").str.len().eq(5).all(), "zcta must always be 5 characters"
    assert df["state_fips"].eq("48").all(), "state_fips must always be 48"
    assert df["puma_geoid"].astype("string").str.len().eq(7).all(), "puma_geoid must always be 7 characters"
    assert df["puma_code"].astype("string").str.len().eq(5).all(), "puma_code must always be 5 characters"
    assert not df.duplicated(["zcta", "puma_geoid"]).any(), "Duplicate zcta + puma_geoid rows found"
    assert df.groupby("zcta")["is_primary_puma_for_zcta"].sum().eq(1).all(), "Every zcta must have exactly one primary PUMA"
    assert_no_missing_required_fields(df, FINAL_COLUMNS)

    area_columns = ["zcta_land_area", "puma_land_area", "overlap_land_area", "overlap_water_area"]
    assert df[area_columns].ge(0).all().all(), "Area values must not be negative"

    zcta_weight_sums = df.groupby("zcta")["zcta_to_puma_weight"].sum()
    if not np.allclose(zcta_weight_sums, 1.0, atol=tolerance):
        bad = zcta_weight_sums.loc[~np.isclose(zcta_weight_sums, 1.0, atol=tolerance)].head(10)
        raise AssertionError(f"zcta_to_puma_weight does not sum to 1.0 for all ZCTAs:\n{bad}")

    puma_weight_sums = df.groupby("puma_geoid")["puma_to_zcta_weight"].sum()
    if not np.allclose(puma_weight_sums, 1.0, atol=tolerance):
        bad = puma_weight_sums.loc[~np.isclose(puma_weight_sums, 1.0, atol=tolerance)]
        if used_filter:
            print(
                "Warning: puma_to_zcta_weight does not sum to 1.0 for every PUMA after target-ZCTA filtering. "
                f"This is expected when some ZCTAs are removed. Affected PUMAs: {len(bad):,}"
            )
        else:
            raise AssertionError(f"puma_to_zcta_weight does not sum to 1.0 for all PUMAs:\n{bad.head(10)}")

    print("Validation complete.")


validate_crosswalk(final_crosswalk, used_target_filter)

## 6. Save Outputs

The main output has the exact requested schema. The audit output keeps the final columns plus the raw Census columns used to build them.

In [ ]:
final_crosswalk.to_csv(MAIN_OUTPUT_PATH, index=False)
audit_crosswalk.to_csv(AUDIT_OUTPUT_PATH, index=False)

print(f"Saved main output: {MAIN_OUTPUT_PATH}")
print(f"Saved audit output: {AUDIT_OUTPUT_PATH}")

## 7. Examples and Summary Statistics

In [ ]:
display(final_crosswalk.head(10))

In [ ]:
example_zctas = ["77494", "77002", "75201", "78701", "79936"]
for example_zcta in example_zctas:
    example_rows = final_crosswalk.loc[final_crosswalk["zcta"].eq(example_zcta)]
    if example_rows.empty:
        print(f"ZCTA {example_zcta}: not present in this crosswalk.")
    else:
        print(f"ZCTA {example_zcta}: {len(example_rows):,} PUMA overlap row(s)")
        display(example_rows)

In [ ]:
zcta_overlap_counts = (
    final_crosswalk.groupby("zcta")
    .size()
    .rename("puma_overlap_count")
    .sort_values(ascending=False)
    .reset_index()
)

display(zcta_overlap_counts.head(20))

In [ ]:
unique_zctas = final_crosswalk["zcta"].nunique()
unique_pumas = final_crosswalk["puma_geoid"].nunique()
average_pumas_per_zcta = zcta_overlap_counts["puma_overlap_count"].mean()
max_pumas_per_zcta = zcta_overlap_counts["puma_overlap_count"].max()
single_puma_share = zcta_overlap_counts["puma_overlap_count"].eq(1).mean()
multi_puma_share = zcta_overlap_counts["puma_overlap_count"].gt(1).mean()

summary_stats = pd.DataFrame(
    {
        "metric": [
            "unique ZCTAs",
            "unique PUMAs",
            "average PUMAs per ZCTA",
            "max PUMAs per ZCTA",
            "percent of ZCTAs that map to exactly one PUMA",
            "percent of ZCTAs that map to multiple PUMAs",
        ],
        "value": [
            unique_zctas,
            unique_pumas,
            round(average_pumas_per_zcta, 3),
            max_pumas_per_zcta,
            round(single_puma_share * 100, 2),
            round(multi_puma_share * 100, 2),
        ],
    }
)

display(summary_stats)

## Final Interpretation

Each output row represents: "This ZCTA overlaps this PUMA by this much land area." Use `zcta_to_puma_weight` as the probability of selecting this PUMA when sampling ACS PUMS profiles for synthetic customers assigned to this ZCTA.